In [57]:
# ==============================================================================
# GLOBAL REPOSITORY CONFIGURATION: VERBOSITY CONTROL
# ==============================================================================
# Set VERBOSE = True to display intermediate circuit diagrams, statevectors,
# and step-by-step register states during circuit execution.
# Set VERBOSE = False for clean, presentation-ready benchmark tables and summaries.
VERBOSE = False

def debug_print(*args, **kwargs):
    """Prints only when global VERBOSE is True."""
    if VERBOSE:
        print(*args, **kwargs)

# For plotting we include this code
# To visualize call viz_circle.
import math
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


# Takes a statevector and a label
class QubitSystem:

  def __init__(
      self,
      statevector: np.ndarray,
      label: str = "Qubit System",
      registers=None,
  ):
    self.label = label
    self.registers = registers  # List of (name, size) tuples
    self.set_statevector(statevector)

  def set_statevector(self, statevector: np.ndarray):
    sv = np.asarray(statevector, dtype=np.complex128).flatten()
    if sv.ndim != 1:
      raise ValueError("Statevector must be 1D.")
    n_states = sv.size
    n_qubits = int(round(math.log2(n_states)))
    if 2**n_qubits != n_states:
      raise ValueError("Length must be a power of 2.")
    norm = np.linalg.norm(sv)
    if norm != 0 and not np.isclose(norm, 1.0):
      sv = sv / norm

    self.n_qubits = n_qubits
    self.n_states = n_states
    self.amps = sv
    self.prob = np.abs(sv) ** 2
    self.phase = np.angle(sv)

  def viz_circle(
      self,
      max_cols: int = 4,
      figsize_scale: float = 2.5,
      threshold: float = 1e-5,
  ):
    # 1. Filter for only the states with non-zero probability
    active_indices = [
        idx for idx in range(self.n_states) if self.prob[idx] > threshold
    ]
    if len(active_indices) == 0:
      print("No active states with amplitude above threshold.")
      return

    # Resolve registers if not explicitly passed
    regs = self.registers
    if not regs:
      if self.n_qubits == 9:
        regs = [("index", 2), ("text", 4), ("pattern", 2), ("flag", 1)]
      elif self.n_qubits == 8:
        regs = [("index", 2), ("text", 4), ("pattern", 2)]
      elif self.n_qubits == 15:
        regs = [("index", 2), ("text", 8), ("pattern", 4), ("flag", 1)]
      elif self.n_qubits == 14:
        regs = [("index", 2), ("text", 8), ("pattern", 4)]

    # Sort active states by index register (lowest bits)
    if regs and any(r[0] == "index" for r in regs):
      idx_size = next(r[1] for r in regs if r[0] == "index")
      active_indices.sort(key=lambda idx: idx % (2**idx_size))
    else:
      active_indices.sort(key=lambda idx: idx % 4)

    num_active = len(active_indices)
    cols = max(1, min(max_cols, num_active))
    rows = int(math.ceil(num_active / cols))

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(cols * figsize_scale, rows * (figsize_scale + 0.2)),
        squeeze=False,
    )

    def format_title(idx: int) -> str:
      b = format(idx, f"0{self.n_qubits}b")
      if not regs:
        return f"|{b}⟩"

      parts = {}
      cursor = 0
      for name, size in reversed(regs):
        parts[name] = b[cursor : cursor + size]
        cursor += size

      title_elements = []
      if "index" in parts:
        title_elements.append(f"|{parts['index']}⟩")
      if "text" in parts:
        title_elements.append(f"|{parts['text'][::-1]}⟩")
      if "pattern" in parts:
        title_elements.append(f"|{parts['pattern'][::-1]}⟩")
      if "flag" in parts:
        title_elements.append(f"|flag={parts['flag']}⟩")
      return "".join(title_elements)

    # 3. Draw each active state
    for i in range(rows * cols):
      r, c = divmod(i, cols)
      ax = axes[r, c]
      ax.set_aspect("equal")
      ax.set_xlim(0, 1)
      ax.set_ylim(0, 1)
      ax.axis("off")

      if i >= num_active:
        ax.set_visible(False)
        continue

      idx = active_indices[i]

      # Outer reference circle
      ax.add_patch(plt.Circle((0.5, 0.5), 0.48, fill=False, lw=1.0, alpha=0.5))

      # Filled disk (area proportional to probability)
      radius = 0.48 * np.sqrt(self.prob[idx])
      ax.add_patch(plt.Circle((0.5, 0.5), radius, alpha=0.25))

      # Phase arrow
      angle = self.phase[idx] + np.pi / 2
      L = 0.45
      x2 = 0.5 + L * np.cos(angle)
      y2 = 0.5 + L * np.sin(angle)
      ax.arrow(
          0.5,
          0.5,
          x2 - 0.5,
          y2 - 0.5,
          head_width=0.03,
          head_length=0.05,
          length_includes_head=True,
      )

      ax.set_title(format_title(idx), fontsize=9)

    fig.suptitle(self.label, fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def plot_state_vector(qc_or_data, label="Statevector", force=False):
  """Plots the statevector using circular phase visualization.
  By default, only plots if VERBOSE is True, unless force=True.
  """
  if not (VERBOSE or force):
    return
  if isinstance(qc_or_data, QuantumCircuit):
    sv_data = Statevector.from_instruction(qc_or_data).data
    reg_info = [(reg.name, reg.size) for reg in qc_or_data.qregs]
    QubitSystem(sv_data, label, registers=reg_info).viz_circle()
  else:
    sv_data = qc_or_data.data if hasattr(qc_or_data, "data") else qc_or_data
    QubitSystem(sv_data, label).viz_circle()


In [58]:
def debug_xor(qc):
    """Prints active states in the register after XOR only when VERBOSE is True."""
    if not VERBOSE:
        return
    # Ordered oppositely |pattern> |text> |shift>
    sv = Statevector.from_instruction(qc)
    for bitstring, amp in sv.to_dict().items():
        if abs(amp) > 1e-5:
            print("bitstring: ", bitstring)
            print("amp: ", amp)
            pattern_part = bitstring[:2][::-1] # Reverse back so they appear in correct order
            text_part = bitstring[2:6][::-1] # Reverse back so they appear in correct order
            shift_part = bitstring[6:8]
            k = int(shift_part, 2)
            print(f"Shift k={k} ({shift_part}): pattern_reg = '{pattern_part}'")


In [59]:
# N is the length of Text pattern
# M is the length of pattern
import math
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector

text: str = "1011"
pattern: str = "01"

DNA_ENCODING = {
    'A': '00',
    'C': '01',
    'G': '10',
    'T': '11'
}

# ACGT --> 00011011, CG = 0110
def encode_dna(sequence: str) -> str:
    """Converts a DNA string (e.g. 'ACGT') into a binary bitstring."""
    test_str = "".join(DNA_ENCODING[base] for base in sequence)
    debug_print("encoded DNA: ", test_str)
    return test_str

# Paper convention: text_reg[i] stores t_i.
# Qiskit displays basis-state labels as |q_(N-1)...q_0|,
# so printed bitstrings appear reversed relative to T = t0 t1 ... t_(N-1).
def transform_text_reg(qc, text: str, text_reg):
    for i, bit in enumerate(text):
        if bit == "1":
            qc.x(text_reg[i])
        debug_print(f"i: {i}, bit: {bit}")

def transform_pattern_reg(qc, pattern: str, pattern_reg):
    for i, bit in enumerate(pattern):
        if bit == "1":
            qc.x(pattern_reg[i])
        debug_print(f"i: {i}, bit: {bit}")

def prepare_input(text: str, pattern: str) -> tuple[QuantumCircuit, QuantumRegister, QuantumRegister, QuantumRegister, QuantumRegister]:
    # T = t0t1t2t3 = 1011
    # t_i <=> q_i,so t0 = q0
    # Paper/logical order: (t0, t1, t2, t3)
    # Qiskit printed ket order: (q3, q2, q1, q0)
    N: int = len(text)
    M: int = len(pattern)
    index_size: int = math.ceil(math.log2(N))

    debug_print("N: ", N)
    debug_print("M: ", M)
    debug_print("i: ", index_size)

    # Index register has n = log2(N) qubits.
    # Text register: N qubits
    # Pattern register: M qubits
    # Index register: log2(N) qubits
    # Text and pattern are prepared as basis states.
    # Only the index register is put into uniform superposition.

    index_reg = QuantumRegister(index_size, "index") # |k>
    text_reg = QuantumRegister(N, "text")            # |T>
    pattern_reg = QuantumRegister(M, "pattern")      # |P>

    # Initialize flag_reg helper
    flag_reg = QuantumRegister(1, "flag")

    qc = QuantumCircuit(index_reg, text_reg, pattern_reg, flag_reg)

    debug_print("------------------")
    transform_text_reg(qc, text, text_reg)
    debug_print("------------------")

    transform_pattern_reg(qc, pattern, pattern_reg)

    # Put index register into uniform superposition
    qc.h(index_reg)
    return qc, index_reg, text_reg, pattern_reg, flag_reg

def prepare_dna_input(text_dna: str, pattern_dna: str) -> tuple[QuantumCircuit, QuantumRegister, QuantumRegister, QuantumRegister, QuantumRegister]:
    L = len(text_dna)
    M_K = len(pattern_dna)
    index_size = math.ceil(math.log2(L))

    index_reg = QuantumRegister(index_size, "index") # |k>
    text_reg = QuantumRegister(2 * L, "text")            # |T>
    pattern_reg = QuantumRegister(2 * M_K, "pattern")      # |P>

    # Initialize flag_reg helper
    flag_reg = QuantumRegister(1, "flag")

    qc = QuantumCircuit(index_reg, text_reg, pattern_reg, flag_reg)

    # Encode binary bits onto text and pattern, e.g., ACGT --> 00011011
    binary_text = encode_dna(text_dna)
    binary_pattern = encode_dna(pattern_dna)

    transform_text_reg(qc, binary_text, text_reg)
    transform_pattern_reg(qc, binary_pattern, pattern_reg)

    # Put index register into uniform superposition
    qc.h(index_reg)
    return qc, index_reg, text_reg, pattern_reg, flag_reg

def verify_hadamard_uniformity(qc: QuantumCircuit, index_reg):
    state = Statevector.from_instruction(qc)
    index_qubits = [qc.find_bit(q).index for q in index_reg]
    debug_print(state.probabilities_dict(qargs=index_qubits))

# Cyclic shift operator with qubits_per_unit
def unified_build_cyclic_shift_operator(index_len, num_text_units, qubits_per_unit=1):
    """
    Dynamically shifts text_reg based on the binary value of index_reg.
    Uses visited list to prevent redundant or undoing gates.
    """
    template_qc = QuantumCircuit(index_len + (num_text_units * qubits_per_unit))
    
    for i in range(index_len):
        shift_amount = (2**i) % num_text_units
        if shift_amount == 0: continue
            
        visited = [False] * num_text_units

        for j in range(num_text_units):
            if not visited[j]:
                current = j
                while True:
                    next_node = (current + shift_amount) % num_text_units
                    if next_node == j:
                        break
                    
                    for offset in range(qubits_per_unit):
                        template_qc.cswap(
                            i, 
                            index_len + (current * qubits_per_unit) + offset, 
                            index_len + (next_node * qubits_per_unit) + offset
                        )
                    visited[next_node] = True
                    current = next_node
                    visited[j] = True

    return template_qc

# DEPRECATED
def build_cyclic_shift_operator(index_len, text_len):
    """
    Dynamically shifts text_reg based on the binary value of index_reg.
    """
    template_qc = QuantumCircuit(index_len + text_len)
    for i in range(index_len):
        shift_amount = (2**i) % text_len
        if shift_amount == 0: continue
        visited = [False] * text_len
        for j in range(text_len):
            if not visited[j]:
                current = j
                while True:
                    next_node = (current + shift_amount) % text_len
                    if next_node == j:
                        break
                    template_qc.cswap(i, index_len + current, index_len + next_node)
                    visited[next_node] = True
                    current = next_node
                visited[j] = True
    return template_qc

# Implement XOR
def apply_xor(qc, text_reg, pattern_reg):
    """Applies the XOR function via controlled NOTs on the pattern register with the text register."""
    for i in range(len(pattern_reg)):
        qc.cx(text_reg[i], pattern_reg[i])


In [60]:
def unmark_invalid_match(qc, index_reg, pattern_reg, flag, invalid_k):
    # Turn the bit pattern for invalid_k into all 1s
    for j in range(len(index_reg)):
        bit = (invalid_k >> j) & 1
        if bit == 0:
            qc.x(index_reg[j])

    # Fires only for this exact index AND an exact pattern match
    qc.mcx(
        list(index_reg) + list(pattern_reg),
        flag
    )

    # Restore index register
    for j in range(len(index_reg)):
        bit = (invalid_k >> j) & 1
        if bit == 0:
            qc.x(index_reg[j])


def grover_oracle(qc: QuantumCircuit, index_reg: QuantumRegister, text_reg: QuantumRegister, pattern_reg: QuantumRegister, flag_reg: QuantumRegister, mode_is_dna: bool = False, verbose: bool = None):
    is_verbose = VERBOSE if verbose is None else verbose

    # 1. Determine unit size (1 for binary, 2 for DNA)
    qubits_per_unit = 2 if mode_is_dna else 1
    num_text_units = len(text_reg) // qubits_per_unit
    num_pat_units = len(pattern_reg) // qubits_per_unit

    cyclic_shift_circuit_template = unified_build_cyclic_shift_operator(len(index_reg), num_text_units, qubits_per_unit)
    qc.compose(cyclic_shift_circuit_template, qubits=list(index_reg) + list(text_reg), inplace=True)
    plot_state_vector(qc, "After applying cyclic shift", force=is_verbose)

    apply_xor(qc, text_reg, pattern_reg)
    debug_xor(qc)

    # ---- Phase kickback ------
    # Initialize flag into |-> state
    if is_verbose:
        print(qc.draw('text', fold=-1))
    qc.x(flag_reg[0]) # |0> --> |1>
    plot_state_vector(qc, "After applying qc.x(flag_reg[0]).", force=is_verbose)
    qc.h(flag_reg[0]) # 1/sqrt(2) (|0> - |1>)
    plot_state_vector(qc, "After applying qc.h(flag_reg[0]).", force=is_verbose)

    # Mark state
    qc.x(pattern_reg) # Flips all qubits of pattern_register: matches become |11..>
    if is_verbose:
        print(qc.draw('text', fold=-1))

    qc.mcx(list(pattern_reg), flag_reg[0]) # Phase kickback

    # Invalid positions flipped back, only consider invalid indices.
    first_invalid = num_text_units - num_pat_units + 1
    for invalid_k in range(first_invalid, 2**len(index_reg)):
        unmark_invalid_match(
            qc,
            index_reg,
            pattern_reg,
            flag_reg[0],
            invalid_k
        )

    plot_state_vector(qc, "After applying toffoli, i.e., |-> to correct branch.", force=is_verbose)
    qc.x(pattern_reg)  # Flip back the pattern register
    plot_state_vector(qc, "After restoring pattern registers.", force=is_verbose)

    # Uncompute the flag
    qc.h(flag_reg[0]) # Double Hadamard yields identity
    qc.x(flag_reg[0])
    plot_state_vector(qc, "After uncomputing the flag.", force=is_verbose)
    if is_verbose:
        print(qc.draw('text', fold=-1))

    # Undo XOR and cyclic shift
    apply_xor(qc, text_reg, pattern_reg)
    qc.compose(cyclic_shift_circuit_template.inverse(), qubits=list(index_reg) + list(text_reg), inplace=True)
    if is_verbose:
        print(qc.draw('text', fold=-1))
    plot_state_vector(qc, "After restoring undoing XOR and cyclic shifts.", force=is_verbose)


### Grover diffuser

In [61]:
# Right now the oracle has marked the state in question. We need to increase its probability
def apply_diffusion(qc, index_reg):
    # 1/2 (|00>|1011>|01> - |01>|1011>|01> + |10>|1011>|01> + |11>|1011>|01>)
    qc.h(index_reg) # Inverse Hadamard, though there is - sign.
    qc.x(index_reg)
    
    # Multi-controlled Z reflection
    qc.h(index_reg[-1])
    qc.mcx(list(index_reg[:-1]), index_reg[-1])
    qc.h(index_reg[-1])
    
    qc.x(index_reg)
    qc.h(index_reg)

In [62]:
# Binary case
qc, index_reg, text_reg, pattern_reg, flag_reg = prepare_input("1011", "01")

if VERBOSE:
    print(qc.draw('text', fold=-1))
    plot_state_vector(qc, force=True)
    print(f"Index register qubits: {len(index_reg)}")
    print("---------------------------------------")

grover_oracle(qc, index_reg, text_reg, pattern_reg, flag_reg, mode_is_dna=False)

if VERBOSE:
    print(qc.draw('text', fold=-1))
    plot_state_vector(qc, "After grover oracle", force=True)

apply_diffusion(qc, index_reg)

if VERBOSE:
    print(qc.draw('text', fold=-1))
    plot_state_vector(qc, "After grover diffusion", force=True)

sv = Statevector.from_instruction(qc)
n_idx = len(index_reg)
probs = {k: 0.0 for k in range(2**n_idx)}
for bitstring, p in sv.probabilities_dict().items():
    k = int(bitstring[-n_idx:], 2)
    probs[k] += p
print(f"Binary Matching Result (Text='{text}', Pattern='{pattern}'):")
for k, p in sorted(probs.items()):
    print(f"  Shift k={k}: Probability = {p:.4f}")


Binary Matching Result (Text='1011', Pattern='01'):
  Shift k=0: Probability = 0.0000
  Shift k=1: Probability = 1.0000
  Shift k=2: Probability = 0.0000
  Shift k=3: Probability = 0.0000


# Controlled Test Generator - Benchmarking framework
###### The following code runs the program for various different inputs, collects the results and compares the classical vs quantum algorithm. Written with the help of Gen AI

In [63]:
def classical_matches(text, pattern):
  L, M = len(text), len(pattern)
  return [i for i in range(L - M + 1) if text[i : i + M] == pattern]


# Check our 3 required cases:
print("Case A ('ACGT', 'TA'):", classical_matches("ACGT", "TA"))  # Expected: []
print("Case B ('ACGT', 'CG'):", classical_matches("ACGT", "CG"))  # Expected: [1]
print("Case C ('ACAC', 'AC'):", classical_matches("ACAC", "AC"))  # Expected: [0, 2]

Case A ('ACGT', 'TA'): []
Case B ('ACGT', 'CG'): [1]
Case C ('ACAC', 'AC'): [0, 2]


In [64]:
def quantum_matches(text, pattern, verbose: bool = None):
  qc, index_reg, text_reg, pattern_reg, flag_reg = prepare_dna_input(
      text, pattern
  )
  grover_oracle(
      qc, index_reg, text_reg, pattern_reg, flag_reg, mode_is_dna=True, verbose=verbose
  )   

  sv = Statevector.from_instruction(qc)

  matches = []
  for bitstring, amp in sv.to_dict().items():
    k = int(bitstring[-len(index_reg) :], 2)
    if amp.real < -0.1 and k not in matches:
      matches.append(k)

  return sorted(matches)


cases = [
    ("Case A", "ACGT", "TA"),
    ("Case B", "ACGT", "CG"),
    ("Case C", "ACAC", "AC"),
]

results = []
for label, text, pattern in cases:
  matches = quantum_matches(text, pattern)
  results.append((label, text, pattern, matches))

print("------------BENCHMARK SUMMARY----------------------")
for label, text, pattern, matches in results:
  print(f"{label} ('{text}', '{pattern}'): {matches}")


------------BENCHMARK SUMMARY----------------------
Case A ('ACGT', 'TA'): []
Case B ('ACGT', 'CG'): [1]
Case C ('ACAC', 'AC'): [0, 2]


In [65]:
import random

def generate_dna_test(
    text_len: int, motif_len: int, num_matches: int, seed: int = 42
):
  """Generates a ground truth based controlled DNA benchmark sequence with exact known matching positions."""
  rng = random.Random(seed)
  bases = ["A", "C", "G", "T"]

  # 1. Pick a random motif
  motif = "".join(rng.choice(bases) for _ in range(motif_len))

  # 2. Pick non-overlapping intended positions
  valid_starts = list(range(text_len - motif_len + 1))
  intended_positions = []

  for _ in range(num_matches):
    if not valid_starts:
      break
    pos = rng.choice(valid_starts)
    intended_positions.append(pos)
    # Remove overlapping positions
    valid_starts = [p for p in valid_starts if abs(p - pos) >= motif_len]

  intended_positions.sort()

  # 3. Generate background and insert motif (retry if accidental matches occur)
  for attempt in range(200):
    text_chars = [rng.choice(bases) for _ in range(text_len)]

    for pos in intended_positions:
      text_chars[pos : pos + motif_len] = list(motif)

    text = "".join(text_chars)

    # 4. Classical check: Must match ONLY the intended positions
    actual_matches = classical_matches(text, motif)
    if actual_matches == intended_positions:
      return text, motif, actual_matches

  raise RuntimeError("Could not find clean background without accidental matches. Try another seed.")

In [66]:
# 1. Create the test case and get the answer key
text, motif, targets = generate_dna_test(
    text_len=4, motif_len=2, num_matches=1, seed=123
)

q_matches = quantum_matches(text, motif)

# Compare to ground truth
print(f"Text: '{text}' | Motif: '{motif}'")
print(f"Expected: {targets}")
print(f"Quantum:  {q_matches}")
print("Result:   ", "PASS" if q_matches == targets else "FAIL")

Text: 'AGAA' | Motif: 'AG'
Expected: [0]
Quantum:  [0]
Result:    PASS


# Iterated Grover Test

In [67]:
# Iterations to test
iterations_to_test = [1, 2, 3]
iteration_results = []

print("Running Grover Iterations (circuit building)...")

for R in iterations_to_test:
  # 1. Prepare circuit
  qc, index_reg, text_reg, pattern_reg, flag_reg = prepare_dna_input(
      "ACGT", "CG"
  )

  # 2. Apply R iterations of (Oracle + Diffuser)
  for _ in range(R):
    grover_oracle(
        qc, index_reg, text_reg, pattern_reg, flag_reg, mode_is_dna=True
    )
    apply_diffusion(qc, index_reg)

  # 3. Measure candidate shift probabilities
  sv = Statevector.from_instruction(qc)
  n_idx = len(index_reg)
  probs = {k: 0.0 for k in range(2**n_idx)}
  for bitstring, p in sv.probabilities_dict().items():
    k = int(bitstring[-n_idx:], 2)
    probs[k] += p

  iteration_results.append((R, probs))

# ----- Table ------
print("\n" + "=" * 65)
print("     GROVER ITERATION STUDY: CANDIDATE SHIFT PROBABILITIES")
print("          Text='ACGT', Motif='CG' (Target shift: k=1)")
print("=" * 65)
print("  R  |  P(k=0)  |  P(k=1) [TARGET]  |  P(k=2)  |  P(k=3)  | Success")
print("-" * 65)
for R, p in iteration_results:
  print(
      f"  {R}  |  {p[0]:6.4f}  |      {p[1]:6.4f}       |  {p[2]:6.4f}  |"
      f"  {p[3]:6.4f}  | {p[1]*100:5.1f}%"
  )
print("=" * 65)


Running Grover Iterations (circuit building)...

     GROVER ITERATION STUDY: CANDIDATE SHIFT PROBABILITIES
          Text='ACGT', Motif='CG' (Target shift: k=1)
  R  |  P(k=0)  |  P(k=1) [TARGET]  |  P(k=2)  |  P(k=3)  | Success
-----------------------------------------------------------------
  1  |  0.0000  |      1.0000       |  0.0000  |  0.0000  | 100.0%
  2  |  0.2500  |      0.2500       |  0.2500  |  0.2500  |  25.0%
  3  |  0.2500  |      0.2500       |  0.2500  |  0.2500  |  25.0%


In [ ]:
cases = [
    ("Case A (No match)", "ACGT", "TA"),
    ("Case B (One match)", "ACGT", "CG"),
    ("Case C (Multiple matches)", "ACAC", "AC"),
]

print("=" * 95)
print(
    f"{'Test Case':<26} | {'Text':<6} | {'Motif':<6} | {'Classical Expected':<20}"
    f" | {'Quantum Result (R=1)':<28}"
)
print("=" * 95)

for label, text, motif in cases:
  # 1. Classical matches
  c_matches = classical_matches(text, motif)
  if not c_matches:
    c_str = "None ([])"
  elif len(c_matches) == 1:
    c_str = f"Shift {c_matches[0]} ([{c_matches[0]}])"
  else:
    c_str = f"Shifts {', '.join(map(str, c_matches))} ({c_matches})"

  # 2. Quantum execution (R=1: Oracle + Diffusion)
  qc, index_reg, text_reg, pattern_reg, flag_reg = prepare_dna_input(
      text, motif
  )
  grover_oracle(
      qc, index_reg, text_reg, pattern_reg, flag_reg, mode_is_dna=True
  )
  apply_diffusion(qc, index_reg)

  # 3. Probabilities across shift index register
  sv = Statevector.from_instruction(qc)
  n_idx = len(index_reg)
  probs = {k: 0.0 for k in range(2**n_idx)}
  for bitstring, p in sv.probabilities_dict().items():
    k = int(bitstring[-n_idx:], 2)
    probs[k] += p

  # 4. Format quantum output with probabilities
  if not c_matches:
    q_str = f"None (Uniform {probs[0]*100:.1f}%)"
  elif len(c_matches) == 1:
    k_target = c_matches[0]
    q_str = f"Shift {k_target} ({probs[k_target]*100:.1f}%)"
  else:
    target_prob = sum(probs[k] for k in c_matches) * 100
    each_prob = probs[c_matches[0]] * 100
    q_str = (
        f"Shifts {', '.join(map(str, c_matches))} ({each_prob:.1f}% each,"
        f" {target_prob:.1f}% tot)"
    )

  print(f"{label:<26} | {text:<6} | {motif:<6} | {c_str:<20} | {q_str:<28}")

print("=" * 95)

Test Case                  | Text   | Motif  | Classical       | {'Quantum Result (R=1)':<25}
----------------------------------------------------------------------------------------
Case A (No match)          | ACGT   | TA     | None ([])       | None (Uniform 25%)       
Case B (One match)         | ACGT   | CG     | Shift 1 ([1])   | Shift 1 (100.0%)         
Case C (Multiple matches)  | ACAC   | AC     | Shifts 0, 2 ([0, 2]) | Shifts 0, 2 (Phase marked)
